In [1]:
import os
import datetime
import yaml
import numpy as np
import pandas as pd
import pathlib
import tensorflow as tf
from operator import itemgetter
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping
import matplotlib.pyplot as plt

from src.train.models import build_dmq_v0
from src.train.losses import make_total_tilted_loss
from src.utils.evaluation import compute_oos_r1_score, compute_oos_r2_score
from src.preprocessing.prepare_quantile_data import prepare_quantile_data

# =====================================================================
# Configuration
# =====================================================================
# Load config for defaults
with open("config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Data configurations
TARGET_IDX = 0  # 0: Infl_yoy, 1: IP_yoy, 2: Unrate_yoy
TIME_STEPS = 12  # Number of quarters for RNN
YEAR = 1997     # Train cutoff year
VAL_YEARS = 5   # Number of validation years

QUANTILES = [0.05, 0.25, 0.5, 0.75, 0.95]
EPOCHS = 200
BATCH_SIZE = 32
LEARNING_RATE = 0.001

target_name_dict = {
    0: 'Infl_yoy',
    1: 'IP_yoy',
    2: 'Unrate_yoy',
}

target_name = target_name_dict[TARGET_IDX]
print(f"Experimental framework for target: {target_name}")

Experimental framework for target: Infl_yoy


In [ ]:
# =====================================================================
# Load and Prepare Data
# =====================================================================
# Setup paths exactly like main_train.py
base_path = pathlib.Path('.')
INPUT_FILES = config['input_files']
TARGET_FILE = config['target_file']
target_path = base_path/ 'data' / 'processed' / TARGET_FILE
input_paths = [base_path / 'data' / 'processed' / file for file in INPUT_FILES]

non_rnn_data, rnn_data, meta_data = prepare_quantile_data(
    target=TARGET_IDX,
    time_steps=TIME_STEPS,
    targets_path=target_path,
    input_paths=input_paths,
    start_date='1961-01-01',
    train_cutoff_year=YEAR,
    n_quantiles=len(QUANTILES),
    val_years=VAL_YEARS
)

# Extract RNN data
(
    mq_y_train_rnn, mq_y_val_rnn, mq_y_train_full_rnn,
    X_train_rnn, X_val_rnn, X_train_full_rnn, X_test_rnn,
) = itemgetter(
    'mq_y_train_rnn', 'mq_y_val_rnn', 'mq_y_train_full_rnn',
    'X_train_rnn', 'X_val_rnn', 'X_train_full_rnn', 'X_test_rnn',
)(rnn_data)

y_test = non_rnn_data['y_test']

print(f"Features shape: {X_train_rnn.shape}")
print(f"Target shape: {mq_y_train_rnn.shape}")

In [ ]:
# =====================================================================
# Build and Train the Model
# =====================================================================
log_dir = "local_tuning_logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)
early_stopping_args = {
    'monitor': 'val_loss',
    'min_delta': 1e-3,
    'patience': 5,
    'restore_best_weights': True,
    'verbose': 0
}
fit_params = {
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'validation_data': (X_val_rnn, mq_y_val_rnn),
    'verbose': 1,
    'shuffle': False
}
early_stopping = EarlyStopping(**early_stopping_args)

# Build Model
model = build_dmq_v0(
    input_shape=X_train_rnn.shape[1:],
    n_recurrent_layers=1,
    n_shared_layers=1,
    n_qtask_layers=1,
    n_recurrent_nodes=32,
    n_shared_nodes=32,
    n_task_nodes=32,
    lr=0.001,
    dropout=0.0,
    rec_drop=0.0,
    recurrent_layer_type='slstm',
    lower_quantiles=[0.05, 0.25], 
    upper_quantiles=[0.75, 0.95]
)

# Fit
history = model.fit(
    X_train_rnn,
    mq_y_train_rnn,
    validation_data=(X_val_rnn, mq_y_val_rnn),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
    callbacks=[early_stopping, tensorboard_callback]
)

# To view tensorboard, run in terminal: tensorboard --logdir local_tuning_logs/fit


In [ ]:
# =====================================================================
# Evaluation and Visualization
# =====================================================================
# Generate predictions for the test set
test_preds = model.predict(X_test_rnn)

# Reshape test_preds if needed (usually shape is (samples, n_quantiles))
if len(test_preds.shape) > 2:
    test_preds = np.squeeze(test_preds)

print(test_preds.shape, y_test.shape)

# Compute out-of-sample R1 and R2 scores
r1_scores = compute_oos_r1_score(
    y_test.values if hasattr(y_test, 'values') else np.array(y_test), 
    test_preds, 
    np.tile(np.mean(mq_y_train_rnn[:, 0]), (len(y_test), len(QUANTILES))), # naive baseline 
    np.array(QUANTILES)
)

r2_scores = compute_oos_r2_score(
    y_test.values if hasattr(y_test, 'values') else np.array(y_test), 
    test_preds, 
    np.tile(np.mean(mq_y_train_rnn[:, 0]), (len(y_test), len(QUANTILES))), # naive baseline
    np.array(QUANTILES)
)

print("-" * 50)
print("Out-of-sample R1 Scores:")
for i, q in enumerate(QUANTILES):
    print(f"Quantile \t{q}: \t{r1_scores[i]:.4f}")
print(f"Mean R1: \t\t{np.mean(r1_scores):.4f}")
print("-" * 50)
print("Out-of-sample R2 Scores:")
for i, q in enumerate(QUANTILES):
    print(f"Quantile \t{q}: \t{r2_scores[i]:.4f}")
print(f"Mean R2: \t\t{np.mean(r2_scores):.4f}")
print("-" * 50)


# Visualization of Quantiles
plt.figure(figsize=(10, 6))

time_axis = np.arange(len(y_test))
plt.plot(time_axis, y_test, label='True Value', color='black', linewidth=2)

colors = plt.cm.Blues(np.linspace(0.3, 1, len(QUANTILES)))
for i, q in enumerate(QUANTILES):
    plt.plot(time_axis, test_preds[:, i], label=f'Quantile {q}', color=colors[i], linestyle='--')

plt.fill_between(time_axis, test_preds[:, 0], test_preds[:, -1], color='blue', alpha=0.1, label='90% Interval')

plt.title(f'Quantile Forecasts Out-of-Sample for {target_name}')
plt.xlabel('Quarters out-of-sample')
plt.ylabel(target_name)
plt.legend(loc='upper right')
plt.grid(True)
plt.show()


In [ ]:
# =====================================================================
# Load and Prepare Data
# =====================================================================
def load_and_prep_data(target, config):
    df = pd.read_csv('data/processed/df_yoy.csv') # Assuming processed data exists here, you might need to adjust path
    # If not, you might need to run main_data.py to get processed data.

    # Normally prepare_quantile_data returns datasets based on config
    # Since we manually tune, let's load what prepare_quantile_data returns:
    # Here is an example of what prepare_quantile_data might do if used directly
    train_data, val_data, test_data, qt, qt_target, target_cols, feature_cols = prepare_quantile_data(
        df=df,
        target_col=target,
        config=config,
        random_state=42
    )
    
    return train_data, val_data, test_data, feature_cols

train_data, val_data, test_data, feature_cols = load_and_prep_data(TARGET_VARIABLE, config)

X_train, y_train = train_data['X'], train_data['y']
X_val, y_val = val_data['X'], val_data['y']
X_test, y_test = test_data['X'], test_data['y']

print(f"Features shape: {X_train.shape}")
print(f"Target shape: {y_train.shape}")